# Demo 4: KV transfer of the task span: where the goal heads read the sub-goal from

**Insight.** During execution, the LM frequently re-reads the goal from the task span using the goal heads. From a pivot point, controlling what the goal heads read from the task span can successfully change the LM's action. 

**Setup.** 
\begin{align*}
p_\mathrm{src}: \mathrm{\underline{ab}c \rightarrow \mathrm{c}} \\
p_\mathrm{dst}: \mathrm{\underline{ab}d \rightarrow \mathrm{d}}
\end{align*}

The source and destination prompts share the same sub-tasks, $\mathrm{a}$ and $\mathrm{b}$, that are prefilled in. But their pending operation is different: the source prompt has $\mathrm{c}$ as the pending operation, while the destination prompt has $\mathrm{d}$.

**Intervention (pivot-onwards KV transfer).** In the patched run, at the pivot token (`" ="`), we intervene on what the 250 goal heads read from the task span. Each attention head reads from its key-value cache. For a goal head, we swap the key-value rows corresponding to the task span in DST with those from SRC. The rest of the rows in the KV cache remain untouched.

The intervention is done at the pivot token, and every subsequent token is generated with the patched KV cache.

If the LM implements the source operation $\mathrm{c}$, instead of the native destination operation $\mathrm{d}$, we say that the intervention is successful and count that as evidence for the re-reading hypothesis.

**Paper.** §5.1.1 (`fig:kv-transfer`), App. `app:transfer:kv` (`fig:kv-barcharts`, `tab:kv-grid`).

Needs one 80 GB GPU (Qwen2.5-Coder-32B-Instruct in bf16). The executed outputs are saved in this file.

**The setup** (schematics from the lab talk):

![Non-Recurrent Goal Seeking (NRGS): at each step, re-read the goal from the task span, compare it with progress, act (§5.1).](figures/04_nrgs.png)

*Non-Recurrent Goal Seeking (NRGS): at each step, re-read the goal from the task span, compare it with progress, act (§5.1).*

![KV transfer: swap only the goal heads' view of the task span, from the pivot on (§5.1.1).](figures/04_kv_transfer_setup.png)

*KV transfer: swap only the goal heads' view of the task span, from the pivot on (§5.1.1).*

![Which states, which heads, when: one GQA group, the goal heads' keys/values over the task span (App. I.1).](figures/04_kv_transfer_detail.png)

*Which states, which heads, when: one GQA group, the goal heads' keys/values over the task span (App. I.1).*

In [1]:
# repo root = first parent dir that contains src/
import os, sys
_d = os.getcwd()
while _d != '/' and not os.path.isdir(os.path.join(_d, 'src')):
    _d = os.path.dirname(_d)
sys.path.insert(0, _d); os.chdir(_d)
print('repo root:', _d)

# no progress bars: keeps tqdm / HF widget outputs out of the saved notebook
os.environ['TQDM_DISABLE'] = '1'
import functools, tqdm.std, tqdm.notebook, transformers, huggingface_hub
for _c in (tqdm.std.tqdm, tqdm.notebook.tqdm):   # tqdm.notebook ignores TQDM_DISABLE, so force disable=True
    _c.__init__ = functools.partialmethod(_c.__init__, disable=True)
transformers.utils.logging.disable_progress_bar()
transformers.utils.logging.set_verbosity_error()      # no generation-config warnings
huggingface_hub.utils.disable_progress_bars()

# as in the paper's figures: source red, destination green, patched run purple
from typing import Literal
ide_theme: Literal['dark', 'light'] = 'dark'
if ide_theme == 'dark':   # light shades; the task-span fill reuses the text color
    SRC_C, DST_C, PAT_C = 'salmon1', 'sea_green3', 'plum2'
    SRC_BG, DST_BG      = SRC_C, DST_C
    PIVOT_BG            = 'bright_yellow'
else:                     # the paper's colors (Figure 5)
    SRC_C, DST_C, PAT_C = '#cc0000', '#38761d', '#5b277d'
    SRC_BG, DST_BG      = '#f4cccc', '#d9ead3'
    PIVOT_BG            = '#ffe6b2'

repo root: /disk/u/arnab/Codes/Projects/goals


In [2]:
CUDA     = '7'
MODEL    = 'Qwen/Qwen2.5-Coder-32B-Instruct'
G250     = 'head_sets/Qwen2.5-Coder-32B-Instruct/goal_topk/goal__lam20_top250.json'
C250     = 'head_sets/Qwen2.5-Coder-32B-Instruct/control/control250__laterhalf.json'
DATATYPE, SRC_LANG, DST_LANG = 'list[str]', 'python', 'cpp'
SRC_NEXT = 'has_palindrome'        # the next sub-task in the SRC prompt
DST_NEXT = 'count_all_lowercase'   # the next sub-task the DST prompt would do on its own
SPAN     = 'task'      # the transplanted window: (1) .. the user turn's <|im_end|>
V_SCALE  = 4.0         # x the transplanted values read by the goal heads (paper: V x 4)
MAX_NEW  = 64

os.environ['CUDA_VISIBLE_DEVICES'] = CUDA

## Load the model
Qwen2.5-Coder-32B-Instruct, the 250 goal heads, and 250 random heads (none of them goal heads) as a control.

In [3]:
import json, torch
from rich.console import Console
from rich.table import Table
from rich.text import Text
from rich.markup import escape

import src.nrgs.data as DATA
import src.nrgs.functional as FUNC
import src.nrgs.kv_transfer as PP

console = Console(width=140, highlight=False)
torch.set_grad_enabled(False)
DATA.CPP_NOTE_IN_SYSTEM = True                     # as in kv_transfer_eval.main()

ctx = FUNC.setup(MODEL, device_map={'': 0})
load = lambda p: [tuple(h) for h in json.load(open(p))]
goal_heads, ctrl_heads = load(G250), load(C250)
ctx.readout_heads, ctx.gsp = goal_heads, None
print(f'goal heads: {len(goal_heads)} | control heads: {len(ctrl_heads)} (overlap {len(set(goal_heads) & set(ctrl_heads))})')

2026-11-03 19:55:18 src.models WARNING  Qwen/Qwen2.5-Coder-32B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory: /disk/u/models


Qwen/Qwen2.5-Coder-32B-Instruct
n_heads=40  n_kv_heads=8  group=5  head_dim=128  n_layer=64
goal heads: 250 | control heads: 250 (overlap 0)


## 1. The contrastive pair
SRC and DST list the same sub-tasks except (3). The task spans are aligned token for token (the shorter prompt is
padded with newlines), so each swapped key/value lands at the same position it had in SRC.

In [4]:
insts, sample_seed = PP.sample_multi_instances(DATATYPE, SRC_LANG, DST_LANG, n_samples=32, seed=1123)
inst = next(i for i in insts if (i.src_frontier.id, i.dst_frontier.id) == (SRC_NEXT, DST_NEXT))
g = PP.build_geometry(inst, span=SPAN, ctx=ctx)

t = Table(title='the two task lists')
t.add_column('#'); t.add_column('SRC task list'); t.add_column('DST task list')
for k, (ps, pd) in enumerate(zip(inst._preds_for(inst.src_frontier), inst._preds_for(inst.dst_frontier)), 1):
    same = ps.id == pd.id
    t.add_row(str(k), escape(ps.description) if same else f'[bold {SRC_C}]{escape(ps.description)}[/]',
              escape(pd.description) if same else f'[bold {DST_C}]{escape(pd.description)}[/]')
console.print(t)
print(f'task span: {len(g["dst_w"])} tokens, at the same positions in both prompts')

                                         the two task lists                                         
┏━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ # ┃ SRC task list                    ┃ DST task list                                             ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1 │ the length of the shortest word  │ the length of the shortest word                           │
│ 2 │ whether any word is empty        │ whether any word is empty                                 │
│ 3 │ whether any word is a palindrome │ the number of words written entirely in lowercase letters │
└───┴──────────────────────────────────┴───────────────────────────────────────────────────────────┘

task span: 32 tokens, at the same positions in both prompts


The two prompts as the model sees them. The highlighted task span is what we transplant: we run only DST, but
the goal heads read SRC's keys and values over this span.

In [5]:
def show(tok, win, title, bg):
    ids = tok['input_ids'][0].tolist()
    start = max(i for i, t in enumerate(ids) if t == ctx.mt.tokenizer.convert_tokens_to_ids('<|im_start|>') and i < min(win))
    txt = Text()
    for i in range(start, len(ids)):
        s = ctx.mt.tokenizer.decode([ids[i]])
        pivot = i == len(ids) - 1                     # the `eq` pivot: the prompt's last token, ` =`
        txt.append(s, style=f'bold black on {PIVOT_BG}' if pivot else f'black on {bg}' if i in win else 'dim')
    console.rule(title); console.print(txt)

show(g['src_tok'], set(g['src_w']), 'SRC prompt (k/v donor)', SRC_BG)
show(g['dst_tok'], set(g['dst_w']), 'DST prompt (the run we patch)', DST_BG)

────────────────────────────────────────────────────────── SRC prompt (k/v donor) ──────────────────────────────────────────────────────────

<|im_start|>user
Write a Python program that implements a function `func` which, given a list of strings `words`, computes:



(1) the length of the shortest word
(2) whether any word is empty
(3) whether any word is a palindrome
<|im_end|>
<|im_start|>assistant
<code>
def func(words: list[str]):
    a = min(len(w) for w in words)
    b = any(len(w) == 0 for w in words)
    c =

────────────────────────────────────────────────────── DST prompt (the run we patch) ───────────────────────────────────────────────────────

<|im_start|>user
Write a C++ program that implements a function `func` which, given a list of strings `words`, computes:
(1) the length of the shortest word
(2) whether any word is empty
(3) the number of words written entirely in lowercase letters
<|im_end|>
<|im_start|>assistant
<code>
auto func(std::vector<std::string> words){
    auto a = INT_MAX;
    for(auto &a_w : words) if((int)a_w.size() < a) a = (int)a_w.size();
    auto b = false;
    for(auto &b_w : words) if(b_w.empty()) b = true;
    auto c =

## 2. Clean runs
Without any intervention, each prompt writes its own sub-task (3), and GoalScope at the pivot ` =` already says
which one it is about to write (as in Demo 1).

In [6]:
stub = lambda prompt: prompt.rsplit('\n', 1)[-1]          # the prefilled `    c =` / `    auto c =`

def show_run(title, color, prompt, gen, readout, verdict=None):
    console.rule(f'[bold {color}]{title}[/]')
    console.print(f'[bold]GoalScope at the pivot:[/] [italic]{escape(readout.strip())}[/]')
    console.print(f'[bold]continuation:[/]\n[grey50]{escape(stub(prompt))}[/][{color}]{escape(gen.rstrip())}[/]')
    if verdict: console.print(f'[bold]judge:[/] {verdict}')
    console.print()

with FUNC.console.capture():                               # silence the per-readout log lines
    clean = PP.clean_runs(g, readout_heads=goal_heads, max_new=MAX_NEW, ctx=ctx)
show_run('SRC run', SRC_C, g['src_prompt'], clean['gen_src'], clean['ro_src'])
show_run('DST run', DST_C, g['dst_prompt'], clean['gen_dst'], clean['ro_dst'])

───────────────────────────────────────────────────────────────── SRC run ──────────────────────────────────────────────────────────────────

GoalScope at the pivot: I'm checking if any element in the list is a palindrome by using a generator expression that checks for any word 
where the word reads the same forwards and backwards.

continuation:
    c = any(w == w[::-1] for w in words)
    return a, b, c
</code>

───────────────────────────────────────────────────────────────── DST run ──────────────────────────────────────────────────────────────────

GoalScope at the pivot: I'm checking if the variable is an integer and if it's greater than 0.

continuation:
    auto c = 0;
    for(auto &c_w : words) if(std::all_of(c_w.begin(), c_w.end(), ::islower)) c++;
    return std::make_tuple(a, b, c);
}
</code>

## 3. Patched runs
Three versions of the DST run, each with the task-span transfer from the pivot on:
- **goal 250, V × 4**: the paper's intervention. The transplanted values are scaled × 4 so SRC's task span
  dominates what the goal heads read.
- **goal 250, V × 1**: the same swap without the boost.
- **control 250, V × 4**: the same swap through 250 random heads. If the effect were just "any heads reading a
  different prompt", this would flip too.

For each run we show the GoalScope readout at the patched pivot, the generation, and the verdict of an LM judge
(Claude): does the code's first computation implement SRC's sub-task (a **hit**) or DST's own, and did the
readout move from DST's sub-task to SRC's?

In [7]:
arms = {
    'goal 250, V x4':  dict(heads=goal_heads, v=V_SCALE),
    'goal 250, V x1':          dict(heads=goal_heads, v=1.0),
    # 'control 250, V x4':       dict(heads=ctrl_heads, v=V_SCALE),
}
with FUNC.console.capture():
    pat = {name: PP.run_patched(g, goal_heads=a['heads'], readout_heads=goal_heads, v_scale=a['v'], max_new=MAX_NEW, ctx=ctx)
           for name, a in arms.items()}
lab = {'src': f'[bold {SRC_C}]c (SRC)[/]', 'dst': f'[bold {DST_C}]d (DST)[/]'}
console.print(f'[bold {SRC_C}]c[/] = {escape(inst.expected_next_desc)}   |   [bold {DST_C}]d[/] = {escape(inst.dst_next_desc)}\n')
for name, p in pat.items():
    causal, ro = PP.judge(inst, dict(src_prompt=g['src_prompt'], dst_prompt=g['dst_prompt'], **clean, **p))
    verdict = (f"code implements {lab.get(causal['label'], causal['label'])} "
               f"({'[bold]HIT[/]' if causal['triggered'] else '[dim]miss[/]'}); "
               f"readout says {lab.get(ro['patched'], ro['patched'])}"
               f"{' ([bold]shifted[/])' if ro['shifted'] else ''}")
    show_run(f'patched DST run: {name}', PAT_C, g['dst_prompt'], p['gen_patched'], p['ro_patched'], verdict)

c = whether any word is a palindrome   |   d = the number of words written entirely in lowercase letters

───────────────────────────────────────────────────── patched DST run: goal 250, V x4 ──────────────────────────────────────────────────────

GoalScope at the pivot: I'm checking if any element in the list is a palindrome.

continuation:
    auto c = false;
    for(auto &c_w : words) if(std::equal(c_w.begin(), c_w.begin() + c_w.size()/2, c_w.rbegin())) c = true;
    return std::make_tuple(a, b, c);
}
</code>

judge: code implements c (SRC) (HIT); readout says c (SRC) (shifted)

───────────────────────────────────────────────────── patched DST run: goal 250, V x1 ──────────────────────────────────────────────────────

GoalScope at the pivot: I'm checking if any element in the list satisfies a condition (e.g., being a palindrome).

continuation:
    auto c = 0;
    for(auto &c_w : words) if(c_w == std::string(c_w.rbegin(), c_w.rend())) c++;
    return std::make_tuple(a, b, c);
}
</code>

judge: code implements c (SRC) (HIT); readout says c (SRC) (shifted)

## What to expect
<img src="figures/04_kv_transfer_results.png" width="400" alt="KV transfer results by language direction (Figure 5, right, in the paper)">

In the patched run, the LM should implement SRC's sub-task instead of its own (causal-shift), and the GoalScope
readout at the pivot should describe SRC's sub-task (readout-shift). Across 512 instances this holds 94% of the time
for the code (90–97% per language direction) and 90% for the readout. The same transfer through 250 random heads
works on 1 of 512. See §5.1.1 and Figure 5 in the paper; App. I.1 (Figure 22, Table 7) breaks it down by datatype.